<a href="https://colab.research.google.com/github/Carlos-Biologist/Automated-_Detection_and_Tracking_Pinnipeds_UAVs_YOLO26_-_BoT-SORT/blob/main/YOLO_v26_IL_Negative_Images.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
  # Force unmounting of any previous Google Drive connection
!fusermount -u /content/drive 2>/dev/null || true

# Import and mount Google Drive
from google.colab import drive
import os
import shutil

try:
    drive.mount('/content/drive', force_remount=True)
except Exception as e:
    # If the local directory already exists incorrectly, remove it before remounting
    if os.path.exists('/content/drive'):
        shutil.rmtree('/content/drive')
    drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


### **YOLO_v26_IL_Negative images**

In [2]:
# ======================================================
# 1. Install and import dependencies
# ======================================================
!pip install ultralytics -qq
!pip install scikit-learn -qq

from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
import ultralytics

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


In [3]:
# ======================================================
# 2. Define directory paths
# ======================================================
base_path = '/content/drive/MyDrive/Colab Notebooks/YOLO_IL_Negativo_ML/YOLO_v26_Negativo/Versão_2'
img_dir = os.path.join(base_path, 'train/images')
lbl_dir = os.path.join(base_path, 'train/labels')

split_dir = os.path.join(base_path, 'split')
train_dir = os.path.join(split_dir, 'train')
val_dir = os.path.join(split_dir, 'val')
test_dir = os.path.join(split_dir, 'test')

# Clean up and recreate destination subdirectories
for d in [train_dir, val_dir, test_dir]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(os.path.join(d, 'images'), exist_ok=True)
    os.makedirs(os.path.join(d, 'labels'), exist_ok=True)

print("📁 Directory structure successfully created!")

📁 Directory structure successfully created!


In [ ]:
# ======================================================
# 3. Read label annotations and filter classes:
# Class 0 = A_AUSTRALIS
# Class 2 = O_FLAVESCENS
# Class 1 = M_LEONINA (excluded)
# Negative images = images with empty label files (or missing files if flagged)
# ======================================================

# Set to True if negative images do not have an associated .txt file
SEM_TXT_COMO_NEGATIVA = False

EXTS = ('.jpg', '.jpeg', '.png')

# stem -> original image filename (retaining correct extension)
img_map = {
    os.path.splitext(f)[0]: f
    for f in os.listdir(img_dir)
    if f.lower().endswith(EXTS)
}

labels_info = []      # Positive images (A. australis / O. flavescens)
neg_info = []         # Negative images
only_leonina = 0      # Images containing exclusively M. leonina (excluded)
mixed_leonina = 0     # Positive images also containing M. leonina (bounding boxes filtered out)

lbl_stems = set()

for lbl_file in os.listdir(lbl_dir):
    if not lbl_file.endswith('.txt'):
        continue

    stem = os.path.splitext(lbl_file)[0]
    lbl_stems.add(stem)

    if stem not in img_map:
        continue

    australis = 0
    flavescens = 0
    leonina = 0

    with open(os.path.join(lbl_dir, lbl_file), 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 0:
                continue

            class_id = int(parts[0])

            if class_id == 0:
                australis += 1
            elif class_id == 1:
                leonina += 1
            elif class_id == 2:
                flavescens += 1

    # --------------------------------------------------
    # Negative: empty annotation file
    # --------------------------------------------------
    if (australis + flavescens + leonina) == 0:
        neg_info.append({'image': img_map[stem]})
        continue

    # --------------------------------------------------
    # Exclusively M. leonina: excluded (not treated as negative)
    # --------------------------------------------------
    if (australis + flavescens) == 0:
        only_leonina += 1
        continue

    if leonina > 0:
        mixed_leonina += 1

    labels_info.append({
        'image': img_map[stem],
        'australis': australis,
        'flavescens': flavescens
    })

# ------------------------------------------------------
# Images without associated .txt files (optional handling)
# ------------------------------------------------------
no_txt = [img_map[s] for s in img_map if s not in lbl_stems]

if SEM_TXT_COMO_NEGATIVA:
    neg_info.extend({'image': img} for img in no_txt)

df = pd.DataFrame(labels_info)
neg_df = pd.DataFrame(neg_info)

# ======================================================
# 4. Identify dominant class per image
# ======================================================
if not df.empty:
    df['dominant_class'] = df[['australis', 'flavescens']].idxmax(axis=1)
    df['dominant_class_id'] = df['dominant_class'].map({
        'australis': 0,
        'flavescens': 1
    })
else:
    df['dominant_class'] = pd.Series(dtype='object')
    df['dominant_class_id'] = pd.Series(dtype='int64')

# Summary prints
print("\n📊 Identified bounding boxes:")
if not df.empty:
    print(df[['australis', 'flavescens']].sum())
else:
    print("No valid positive bounding boxes found.")

print(f"\n🖼️ Valid positive images: {len(df)}")
print(f"🌫️ Negative images: {len(neg_df)}")
print(f"🚫 Images containing exclusively M. leonina (excluded): {only_leonina}")
print(f"⚠️ Positive images also containing M. leonina (boxes filtered out): {mixed_leonina}")
print(f"❓ Images without .txt file: {len(no_txt)}"
      f"{' (included as negatives)' if SEM_TXT_COMO_NEGATIVA else ' (ignored)'}")

In [5]:
# ======================================================
# 4. Dataset Partitioning (70% Train / 20% Val / 10% Test)
# ======================================================

# Positive images: stratified by dominant class
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df['dominant_class_id'],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=1/3,
    stratify=temp_df['dominant_class_id'],
    random_state=42
)

# Negative images: identical proportions, unstratified (no class label assigned)
neg_train, neg_temp = train_test_split(
    neg_df,
    test_size=0.30,
    random_state=42
)

neg_val, neg_test = train_test_split(
    neg_temp,
    test_size=1/3,
    random_state=42
)

In [ ]:
# ======================================================
# 5. Copy images and regenerate annotation files
# ======================================================

def move_files(subset_df, target_dir):
    """
    Copies images to destination folders and regenerates annotation files,
    remapping class IDs and removing excluded species (M. leonina).
    """
    for _, row in subset_df.iterrows():
        img_name = row['image']
        img_src = os.path.join(img_dir, img_name)

        label_name = os.path.splitext(img_name)[0] + '.txt'
        lbl_src = os.path.join(lbl_dir, label_name)

        # ----------------------------------------------
        # Copy image file
        # ----------------------------------------------
        if os.path.exists(img_src):
            shutil.copy(
                img_src,
                os.path.join(target_dir, 'images', img_name)
            )

        # ----------------------------------------------
        # Create updated label file
        # (Negative images result in empty .txt files)
        # ----------------------------------------------
        new_lines = []

        if os.path.exists(lbl_src):
            with open(lbl_src, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 0:
                        continue

                    class_id = int(parts[0])

                    # A. australis remains class 0
                    if class_id == 0:
                        new_lines.append(line)

                    # M. leonina excluded (formerly class 1)
                    elif class_id == 1:
                        continue

                    # O. flavescens remapped: class 2 -> class 1
                    elif class_id == 2:
                        parts[0] = '1'
                        new_lines.append(' '.join(parts) + '\n')

        with open(os.path.join(target_dir, 'labels', label_name), 'w') as f:
            f.writelines(new_lines)


# ======================================================
# Populate subsets (positive + negative images)
# ======================================================

move_files(train_df, train_dir)
move_files(val_df, val_dir)
move_files(test_df, test_dir)

move_files(neg_train, train_dir)
move_files(neg_val, val_dir)
move_files(neg_test, test_dir)

n_total = len(df) + len(neg_df)

print(f"\n📂 Training set:   {len(train_df)} positive + {len(neg_train)} negative "
      f"= {len(train_df) + len(neg_train)} ({(len(train_df) + len(neg_train))/n_total*100:.1f}%)")
print(f"📂 Validation set: {len(val_df)} positive + {len(neg_val)} negative "
      f"= {len(val_df) + len(neg_val)} ({(len(val_df) + len(neg_val))/n_total*100:.1f}%)")
print(f"📂 Test set:       {len(test_df)} positive + {len(neg_test)} negative "
      f"= {len(test_df) + len(neg_test)} ({(len(test_df) + len(neg_test))/n_total*100:.1f}%)")

In [ ]:
# ======================================================
# 6. Quantify final bounding boxes and images per split
# ======================================================

def count_split(split_path):
    """
    Counts total images, positive images, negative images,
    and class-specific bounding boxes within a designated split.
    """
    img_path = os.path.join(split_path, 'images')
    lbl_path = os.path.join(split_path, 'labels')

    class_counts = Counter()
    n_pos = 0
    n_neg = 0

    for lbl_file in os.listdir(lbl_path):
        if not lbl_file.endswith('.txt'):
            continue

        n_boxes = 0

        with open(os.path.join(lbl_path, lbl_file), 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 0:
                    continue

                class_counts[int(parts[0])] += 1
                n_boxes += 1

        if n_boxes > 0:
            n_pos += 1
        else:
            n_neg += 1

    n_img = len([
        f for f in os.listdir(img_path)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])

    # Integrity verification: every image must possess a corresponding .txt file
    if n_img != (n_pos + n_neg):
        print(f"⚠️ Warning in {split_path}: {n_img} images found, but {n_pos + n_neg} labels detected.")

    return class_counts, n_pos, n_neg, n_img


train_counts, train_pos, train_neg, train_img = count_split(train_dir)
val_counts, val_pos, val_neg, val_img = count_split(val_dir)
test_counts, test_pos, test_neg, test_img = count_split(test_dir)

# ======================================================
# Summary DataFrame
# ======================================================

df_counts = pd.DataFrame({
    'Split': ['Training', 'Validation', 'Test'],
    'Total images': [train_img, val_img, test_img],
    'Positive images': [train_pos, val_pos, test_pos],
    'Negative images': [train_neg, val_neg, test_neg],
    'A. australis (class 0)': [
        train_counts[0],
        val_counts[0],
        test_counts[0]
    ],
    'O. flavescens (class 1)': [
        train_counts[1],
        val_counts[1],
        test_counts[1]
    ]
})

# Append summary total row
total = df_counts.drop(columns='Split').sum()
df_counts.loc[len(df_counts)] = ['Total'] + total.tolist()

print("\n📊 Summary of images and bounding boxes per split:")
print(df_counts.to_string(index=False))

In [ ]:
# ======================================================
# 7. Data Visualisation
# ======================================================

plot_df = df_counts[df_counts['Split'] != 'Total'].set_index('Split')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ------------------------------------------------------
# Panel A: Positive and negative images
# ------------------------------------------------------
plot_df[['Positive images', 'Negative images']].plot(
    kind='bar',
    stacked=True,
    ax=axes[0],
    color=['#2a9d8f', '#e9c46a'],
    edgecolor='black'
)

axes[0].set_title('(A) Images per dataset split')
axes[0].set_ylabel('Number of images')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=0)

# Centred label for positive images (bottom segment)
axes[0].bar_label(axes[0].containers[0], label_type='center', fontsize=9)

# Position label for negative images above stacked bar
y_max = (plot_df['Positive images'] + plot_df['Negative images']).max()

for i, (pos, neg) in enumerate(zip(plot_df['Positive images'], plot_df['Negative images'])):
    axes[0].text(
        i, pos + neg + y_max * 0.015,
        f'{int(neg)} negative',
        ha='center', va='bottom',
        fontsize=9, color='#b5851a', fontweight='bold'
    )

# Upper limit buffer to prevent text clipping
axes[0].set_ylim(0, y_max * 1.12)

# ------------------------------------------------------
# Panel B: Bounding boxes per taxon
# ------------------------------------------------------
plot_df[['A. australis (class 0)', 'O. flavescens (class 1)']].plot(
    kind='bar',
    ax=axes[1],
    color=['#264653', '#e76f51'],
    edgecolor='black'
)

axes[1].set_title('(B) Bounding boxes by species')
axes[1].set_ylabel('Number of bounding boxes')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)

for container in axes[1].containers:
    axes[1].bar_label(container, fontsize=9, padding=2)

plt.suptitle('Distribution of aerial images and target classes after filtering', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(base_path, 'dataset_split_distribution.png'), dpi=300, bbox_inches='tight')
plt.show()

In [9]:
# ======================================================
# 8. Configuration, File Paths, and Dataset YAML
# ======================================================
import glob
import json
import re

RESULTS_DIR = '/content/drive/MyDrive/Colab Notebooks/YOLO_IL_Negativo_ML/YOLO_v26_Negativo/Resultados_REVIS_IL'
os.makedirs(RESULTS_DIR, exist_ok=True)

CLASS_NAMES = ['A_AUSTRALIS', 'O_FLAVESCENS']
EPOCHS = 50
IMGSZ = 640
EVAL_TREINO_POR_EPOCA = True   # Compute training mAP per epoch (increases runtime)

# Primary YAML dataset configuration
yaml_file_name = os.path.join(base_path, 'yolov26_config.yaml')
yaml_content = f"""
path: {split_dir}
train: train/images
val: val/images
test: test/images
nc: 2
names:
  0: A_AUSTRALIS
  1: O_FLAVESCENS
"""
with open(yaml_file_name, 'w') as f:
    f.write(yaml_content.strip())

# Auxiliary YAML: redirects validation to training split to evaluate training mAP
yaml_train_eval = os.path.join(base_path, 'yolov26_config_train_eval.yaml')
with open(yaml_train_eval, 'w') as f:
    f.write(yaml_content.replace('val: val/images', 'val: train/images').strip())

print(f"✅ YAML configurations created\n📁 Destination directory: {RESULTS_DIR}")

✅ YAML configurations created
📁 Destination directory: /content/drive/MyDrive/Colab Notebooks/YOLO_IL_Negativo_ML/YOLO_v26_Negativo/Resultados_REVIS_IL


In [ ]:
# ======================================================
# 9. Model Training (YOLO26m)
# ======================================================
model = YOLO('yolo26m.pt')

print(f"\n🚀 Initialising training process (Ultralytics {ultralytics.__version__})...")

results = model.train(
    data=yaml_file_name,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=8,
    seed=42,
    deterministic=True,
    project=RESULTS_DIR,
    name='treino',
    exist_ok=True,
    save_period=1 if EVAL_TREINO_POR_EPOCA else -1,
    plots=True
)

run_dir = str(model.trainer.save_dir)
best_pt = os.path.join(run_dir, 'weights', 'best.pt')
print(f"\n✅ Training complete!\n📁 Output saved to: {run_dir}")

In [ ]:
# ======================================================
# 10. Training vs Validation Performance Curves per Epoch
# ======================================================
res = pd.read_csv(os.path.join(run_dir, 'results.csv'))
res.columns = res.columns.str.strip()
epochs = res['epoch'].astype(int).values

# ------------------------------------------------------
# Training mAP per epoch (evaluates each checkpoint on training set)
# ------------------------------------------------------
train_map50, train_map5095 = None, None

if EVAL_TREINO_POR_EPOCA:
    ckpts = sorted(
        glob.glob(os.path.join(run_dir, 'weights', 'epoch*.pt')),
        key=lambda p: int(re.findall(r'epoch(\d+)\.pt', p)[0])
    )
    train_map50, train_map5095 = [], []

    for p in ckpts:
        m = YOLO(p).val(
            data=yaml_train_eval,
            split='val',
            imgsz=IMGSZ,
            plots=False,
            verbose=False,
            project=RESULTS_DIR,
            name='tmp_eval',
            exist_ok=True
        )
        train_map50.append(m.box.map50)
        train_map5095.append(m.box.map)

    # The i-th checkpoint corresponds to the i-th row in results.csv
    n = min(len(ckpts), len(res))
    res.loc[:n - 1, 'train_mAP50'] = train_map50[:n]
    res.loc[:n - 1, 'train_mAP50-95'] = train_map5095[:n]

    # Free storage space in Google Drive
    for p in ckpts:
        os.remove(p)
    shutil.rmtree(os.path.join(RESULTS_DIR, 'tmp_eval'), ignore_errors=True)

res.to_csv(os.path.join(RESULTS_DIR, 'metrics_per_epoch.csv'), index=False)

# ------------------------------------------------------
# Precision curves (mAP50 and mAP50-95)
# ------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(epochs, res['metrics/mAP50(B)'], '-o', ms=4, color='#1f77b4', label='mAP50 – Validation')
ax.plot(epochs, res['metrics/mAP50-95(B)'], '-o', ms=4, color='#ff7f0e', label='mAP50-95 – Validation')

if EVAL_TREINO_POR_EPOCA:
    ax.plot(epochs, res['train_mAP50'], '--s', ms=4, color='#1f77b4', alpha=0.7, label='mAP50 – Training')
    ax.plot(epochs, res['train_mAP50-95'], '--s', ms=4, color='#ff7f0e', alpha=0.7, label='mAP50-95 – Training')

ax.set_xlabel('Epoch')
ax.set_ylabel('mAP')
ax.set_title('mAP across epochs: training vs validation')
ax.set_ylim(0, 1)
ax.grid(linestyle='--', alpha=0.5)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'mAP_curve_per_epoch.png'), dpi=300, bbox_inches='tight')
plt.show()

# ------------------------------------------------------
# Training vs Validation loss curves (overfitting diagnosis)
# ------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(epochs, res['train/box_loss'], '-o', ms=4, label='box_loss – Training')
ax.plot(epochs, res['val/box_loss'], '-o', ms=4, label='box_loss – Validation')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Box loss across epochs')
ax.grid(linestyle='--', alpha=0.5)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'loss_curve_per_epoch.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ======================================================
# 11. Test Set Evaluation
# ======================================================
model_best = YOLO(best_pt)

# ------------------------------------------------------
# Confidence threshold determined on VALIDATION split (maximum macro F1-score)
# and applied directly to the test set without tuning on test data
# ------------------------------------------------------
mv = model_best.val(
    data=yaml_file_name,
    split='val',
    imgsz=IMGSZ,
    plots=False,
    verbose=False,
    project=RESULTS_DIR,
    name='tmp_val',
    exist_ok=True
)

f1_mean_val = mv.box.f1_curve.mean(0)
from ultralytics.utils.metrics import smooth   # Standard smoothing function employed by Ultralytics

best_conf = float(mv.box.px[smooth(f1_mean_val, 0.1).argmax()])
shutil.rmtree(os.path.join(RESULTS_DIR, 'tmp_val'), ignore_errors=True)

print(f"🎯 Selected confidence threshold (maximum validation F1-score): {best_conf:.3f}")

# ------------------------------------------------------
# Test Set Inference and Evaluation
# ------------------------------------------------------
mt = model_best.val(
    data=yaml_file_name,
    split='test',
    imgsz=IMGSZ,
    plots=True,                                # Generates confusion matrix, PR curves, F1 curves, etc.
    project=RESULTS_DIR,
    name='teste',
    exist_ok=True
)

b = mt.box
idx = int(np.abs(b.px - best_conf).argmin())

# ------------------------------------------------------
# Performance Metrics Table (Precision, Recall, and F1 at validation threshold)
# ------------------------------------------------------
t_img = os.path.join(test_dir, 'images')

rows = []
for k, c in enumerate(b.ap_class_index):
    rows.append({
        'Class': CLASS_NAMES[c],
        'mAP50': b.ap50[k],
        'mAP50-95': b.ap[k],
        'Precision': b.p_curve[k, idx],
        'Recall': b.r_curve[k, idx],
        'F1-Score': b.f1_curve[k, idx],
    })

# Compute macro-average row ('all')
rows.insert(0, {
    'Class': 'all',
    'mAP50': b.map50,
    'mAP50-95': b.map,
    'Precision': np.mean([r['Precision'] for r in rows]),
    'Recall': np.mean([r['Recall'] for r in rows]),
    'F1-Score': np.mean([r['F1-Score'] for r in rows]),
})

df_test = pd.DataFrame(rows)
df_test.to_csv(os.path.join(RESULTS_DIR, 'metricas_teste.csv'), index=False, encoding='utf-8-sig')

info = {
    'confidence_threshold': best_conf,
    'num_test_images': len(os.listdir(t_img)),
}
with open(os.path.join(RESULTS_DIR, 'metricas_teste_info.json'), 'w') as f:
    json.dump(info, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 70)
print("📈 TEST SET PERFORMANCE METRICS")
print("=" * 70)
print(df_test.round(4).to_string(index=False))

# ------------------------------------------------------
# Precision–Recall Curve Visualisation
# ------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 6))
colours = ['#264653', '#e76f51']

for k, c in enumerate(b.ap_class_index):
    ax.plot(
        b.px,
        b.prec_values[k],
        color=colours[c],
        lw=2,
        label=f'{CLASS_NAMES[c]} (AP50 = {b.ap50[k]:.3f})'
    )

ax.plot(
    b.px,
    b.prec_values.mean(0),
    color='black',
    lw=3,
    label=f'All classes (mAP50 = {b.map50:.3f})'
)

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
ax.set_title('Precision–Recall Curve (Independent Test Set)')
ax.grid(linestyle='--', alpha=0.5)
ax.legend(loc='lower left')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'curva_precision_recall_teste.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✅ All outputs successfully saved to: {RESULTS_DIR}")

In [ ]:
# ======================================================
# 12. Visual Inference on Independent Test Set
# ======================================================
# Fallback to default 0.4 if best_conf was not previously defined in runtime
CONF_PRED = best_conf if 'best_conf' in locals() else 0.4
IOU_PRED = 0.6

model_best = YOLO(best_pt)
pred_dir = os.path.join(RESULTS_DIR, 'test_predictions')
os.makedirs(pred_dir, exist_ok=True)

print(f"\n🎯 Exporting test set predictions to: {pred_dir}")

rows = []
stream = model_best.predict(
    source=os.path.join(test_dir, 'images'),  # Includes negative control samples
    conf=CONF_PRED,
    iou=IOU_PRED,
    imgsz=IMGSZ,
    save=True,
    project=RESULTS_DIR,
    name='test_predictions',
    exist_ok=True,
    stream=True,
    verbose=False
)

for r in stream:
    cls = r.boxes.cls.cpu().numpy().astype(int)
    rows.append({
        'image': os.path.basename(r.path),
        'A_AUSTRALIS': int((cls == 0).sum()),
        'O_FLAVESCENS': int((cls == 1).sum())
    })

# Export frame-level detections (verifies false positive rates on negative images)
df_pred = pd.DataFrame(rows)
df_pred.to_csv(os.path.join(pred_dir, 'detections_per_test_image.csv'), index=False)

print(f"\n✅ {len(df_pred)} images successfully processed")
print(f"📸 Annotated test images saved to: {pred_dir}")

In [14]:
# ======================================================
# 13. Checkpoint Verification
# ======================================================
best_model_path = best_pt
assert os.path.exists(best_model_path), f"❌ Checkpoint best.pt not found: {best_model_path}"
print(f"✅ Optimal checkpoint verified: {best_model_path}")

✅ Optimal checkpoint verified: /content/drive/MyDrive/Colab Notebooks/YOLO_IL_Negativo_ML/YOLO_v26_Negativo/Resultados_REVIS_IL/treino/weights/best.pt


In [ ]:
  # ======================================================
# 14. BoT-SORT: Multi-Object Tracking & Individual Counting
# ======================================================
import cv2
import yaml

video_input_path = '/content/drive/MyDrive/Colab Notebooks/YOLO/DJI 0551_Teste contagem.mp4'

CONF_DET = 0.1             # Low confidence threshold: enables BoT-SORT secondary association
MIN_LEN, MIN_CONF = 5, 0.3 # Valid track criterion: >= MIN_LEN frames and mean confidence >= MIN_CONF

BOTSORT_BASE = dict(
    tracker_type='botsort',
    track_high_thresh=0.5,       # Primary high-confidence association threshold
    track_low_thresh=0.1,        # Secondary low-confidence association threshold
    new_track_thresh=0.6,        # Confidence threshold required to initialise a new track
    track_buffer=60,             # Number of frames an unmatched track is retained in memory
    match_thresh=0.8,            # Spatial matching threshold (IoU)
    fuse_score=True,
    gmc_method='sparseOptFlow',  # Camera motion compensation via sparse optical flow
    proximity_thresh=0.5,
    appearance_thresh=0.25,
    with_reid=False,             # Operates in motion-only mode (no visual ReID features)
    model='auto'
)


def save_yaml(cfg, path):
    """Serialises tracker configuration to a local YAML file."""
    with open(path, 'w') as f:
        yaml.safe_dump(cfg, f)
    return path


def count_tracks(df_tr, min_len=5, min_conf=0.3):
    """
    Quantifies unique individuals: tracks with >= min_len frames and mean confidence >= min_conf.
    The species class assigned to each trajectory is determined by majority vote.
    """
    zeros = pd.Series([0, 0], index=[0, 1])
    if df_tr.empty:
        return zeros, pd.DataFrame()

    tracks = df_tr.groupby('id').agg(
        n_frames=('frame', 'count'),
        mean_conf=('conf', 'mean'),
        species_class=('cls', lambda s: s.mode().iloc[0]),
        start_frame=('frame', 'min'),
        end_frame=('frame', 'max')
    ).reset_index()

    valid = tracks[(tracks['n_frames'] >= min_len) & (tracks['mean_conf'] >= min_conf)]
    counts = valid['species_class'].value_counts().reindex([0, 1], fill_value=0)
    return counts, tracks


def run_botsort(video, cfg, run_name, conf_det=0.1, save_video=True):
    """
    Executes BoT-SORT multi-object tracking over input video footage using YOLO26m detections.
    """
    yml = save_yaml(cfg, os.path.join(RESULTS_DIR, f'{run_name}.yaml'))
    m = YOLO(best_model_path)   # Instantiate fresh model to reset internal tracker state

    stream = m.track(
        source=video,
        tracker=yml,
        conf=conf_det,
        iou=0.6,
        imgsz=IMGSZ,
        save=save_video,
        project=RESULTS_DIR,
        name=run_name,
        exist_ok=True,
        stream=True,
        verbose=False
    )

    track_rows = []
    for i, r in enumerate(stream):
        if r.boxes.id is None:
            continue
        ids = r.boxes.id.cpu().numpy().astype(int)
        cls = r.boxes.cls.cpu().numpy().astype(int)
        cf = r.boxes.conf.cpu().numpy()
        for t, c, s in zip(ids, cls, cf):
            track_rows.append((i, int(t), float(s), int(c)))

    return pd.DataFrame(track_rows, columns=['frame', 'id', 'conf', 'cls'])


# ------------------------------------------------------
# Run tracker on survey video footage
# ------------------------------------------------------
df_tr = run_botsort(video_input_path, BOTSORT_BASE, 'video_botsort', conf_det=CONF_DET)

out = os.path.join(RESULTS_DIR, 'video_botsort')
os.makedirs(out, exist_ok=True)

if df_tr.empty:
    raise RuntimeError("⚠️ No detections tracked. Check input video path, weights, or lower CONF_DET.")

counts, tracks = count_tracks(df_tr, MIN_LEN, MIN_CONF)

df_tr.to_csv(os.path.join(out, 'detections_with_id.csv'), index=False)
tracks.to_csv(os.path.join(out, 'trajectories.csv'), index=False)

print("\n🐾 Quantified individuals (validated trajectories):")
print(f"A. australis : {counts[0]}")
print(f"O. flavescens: {counts[1]}")
print(f"Total trajectories: {len(tracks)} | Filtered out by criteria: {len(tracks) - counts.sum()}")

# ------------------------------------------------------
# Export parameter metadata (for Methodology documentation)
# ------------------------------------------------------
with open(os.path.join(out, 'botsort_parameters.json'), 'w') as f:
    json.dump({
        'tracker_config': BOTSORT_BASE,
        'detection_conf_threshold': CONF_DET,
        'min_track_length_frames': MIN_LEN,
        'min_mean_confidence': MIN_CONF,
        'final_counts': {
            'A_AUSTRALIS': int(counts[0]),
            'O_FLAVESCENS': int(counts[1])
        }
    }, f, indent=2)

# ------------------------------------------------------
# Filter sensitivity analysis (computed without reprocessing video)
# ------------------------------------------------------
sens = []
for ml in (1, 3, 5, 10, 15, 30):
    for mc in (0.1, 0.3, 0.5):
        c, _ = count_tracks(df_tr, ml, mc)
        sens.append({
            'min_len': ml,
            'min_conf': mc,
            'A_AUSTRALIS': int(c[0]),
            'O_FLAVESCENS': int(c[1]),
            'total': int(c.sum())
        })

df_sens = pd.DataFrame(sens)
df_sens.to_csv(os.path.join(out, 'filter_sensitivity_analysis.csv'), index=False)
print("\n🔎 Filter sensitivity evaluation (compare against human counts):")
print(df_sens.to_string(index=False))

# ------------------------------------------------------
# Time-series visualization: per-frame count vs cumulative unique tracks
# ------------------------------------------------------
cap = cv2.VideoCapture(video_input_path)
fps_v = cap.get(cv2.CAP_PROP_FPS)
cap.release()

valid_ids = tracks[(tracks['n_frames'] >= MIN_LEN) & (tracks['mean_conf'] >= MIN_CONF)]['id']
dfv = df_tr[df_tr['id'].isin(valid_ids)]

if dfv.empty:
    print("⚠️ No valid tracks available for time-series plotting under current filters.")
else:
    per_frame = dfv.groupby('frame')['id'].nunique()
    first_occurrence = dfv.groupby('id')['frame'].min().value_counts().sort_index().cumsum()
    first_occurrence = first_occurrence.reindex(range(int(df_tr['frame'].max()) + 1)).ffill().fillna(0)

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(per_frame.index / fps_v, per_frame.values, label='Active individuals per frame', alpha=0.7)
    ax.plot(first_occurrence.index / fps_v, first_occurrence.values, label='Cumulative unique individuals', lw=2)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Number of individuals')
    ax.set_title('Pinniped Multi-Object Tracking via BoT-SORT (Motion-Only)')
    ax.grid(alpha=0.4)
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(out, 'botsort_trajectory_counts.png'), dpi=300, bbox_inches='tight')
    plt.show()